# A Study on Vectorizing Ancient Manuscripts: Methodologies and Use Cases

### Part 1 — Turning Verses and Their Commentary into Numbers (LSTM Encoders)

**How the data was collected:** the verse–commentary pairs used across this project were gathered two different ways. Some classical Tamil texts were already available in digitised form and were **web-scraped** directly from online sources — Naaladiyar and Thirukadukam from Project Madurai and the Tamil Virtual University, and the Tholkappiyam sections from similar digitised editions. For material that only existed as scanned printed pages, **OCR** (optical character recognition, via Google Cloud Vision and Surya) was also attempted, to recover verse–commentary text from a scanned Balasundaram commentary edition that had no digital version at all. In the end, OCR output on classical Tamil's dense diacritics and ligatures wasn't accurate enough to trust for this analysis, so the dataset used everywhere below is the **web-scraped data only** — the more reliable of the two collection methods.

**Real counts used across every analysis in this notebook:**

| Text | Pairs |
|---|---|
| Naaladiyar | 393 |
| Tholkappiyam Ezhuthadhikaram | 379 |
| Tholkappiyam Sollathikaram | 287 |
| Tholkappiyam Porulathikaram | 103 |
| Thirukadukam | 100 |
| **Total** | **1,262** |

**What's happening here:** two separate LSTM models — one that only ever reads *verses*, one that only ever reads *urai* (the traditional commentary) — are trained per text (Naaladiyar, Thirukadukam, Tholkappiyam), plus one pair trained on everything combined. That's **8 models total**. Neither model is ever told anything about its counterpart; each learns purely by trying to guess masked-out words in its own text ("fill in the blank" — masked language modelling).

Every trained model turns a sentence into two kinds of number-vector: the **last hidden state** (its final memory after reading the whole sentence) and **mean pooling** (an average of what it "thought" at every word). Both get tested the same five ways below:

1. **Cosine similarity + Euclidean distance** — do a verse's numbers and its own commentary's numbers naturally land close together?
2. **CCA** — is there a straight-line (linear) relationship between the verse-space and the urai-space?
3. **KCCA** — is there a bent/curved (non-linear) relationship instead?
4. **Siamese-style pair classifier** — can a small extra network learn to tell a *real* verse-urai match from a *fake* one?
5. **t-SNE** — a 2D map of all these vectors, to see visually what clusters together.

No train/test split is used here — this is a representation *analysis*, not a benchmark.

## v2 changes (corrected dataset rerun)
- **Data**: loads the corrected corpus from local `data/` (extract `classical_tamil_verse_urai_corpus.zip` there) --- **1,262 pairs, 393 Naaladiyar** (verse #398 explanation fixed). A sanity-gate cell asserts these counts before anything trains.
- **Outputs**: written to local `outputs/` instead of `/kaggle/working/`.
- Seed unchanged (3407) so results are comparable to the v1 (1,261-pair) run.
- **Known caveat carried from v1, addressed in `00_controls_and_baselines.ipynb`:** the raw CCA correlations reported here for Naaladiyar and Thirukadukam are **degenerate** (sample count <= representation dimension makes perfect canonical correlation achievable for *any* data, including random noise). Run notebook 00 first and report PCA-reduced CCA alongside these numbers.

In [ ]:
import json, math, random, re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE
from sklearn.cross_decomposition import CCA
from sklearn.metrics.pairwise import rbf_kernel
from tqdm.auto import tqdm

SEED = 3407
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# --- v3: quiet, reproducible execution ---------------------------------------
import os, warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm as _tqdm
_QUIET = os.environ.get("NB_VERBOSE", "0") != "1"
def tqdm(iterable=None, *args, **kwargs):
    """Silent under nbconvert; set NB_VERBOSE=1 to get bars back."""
    kwargs.setdefault("disable", _QUIET)
    kwargs.setdefault("leave", False)
    return _tqdm(iterable, *args, **kwargs)


In [ ]:
import os, subprocess
from matplotlib.font_manager import FontProperties

def configure_tamil_font():
    # On Kaggle/Linux: install Noto Tamil fonts if missing
    if os.path.exists("/kaggle"):
        try:
            subprocess.run(
                ["apt-get", "install", "-y", "fonts-noto", "fonts-noto-extra",
                 "fonts-lohit-taml"],
                capture_output=True, check=False, timeout=60,
            )
            fm._load_fontmanager(try_read_cache=False)
        except Exception:
            pass

    FONT_FILE_CANDIDATES = [
        "/usr/share/fonts/truetype/noto/NotoSansTamil-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSerifTamil-Regular.ttf",
        "/usr/share/fonts/truetype/lohit-tamil/Lohit-Tamil.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSerif.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "C:/Windows/Fonts/nirmala.ttf",
        "C:/Windows/Fonts/latha.ttf",
        "C:/Windows/Fonts/arialuni.ttf",
    ]
    font_file = next((p for p in FONT_FILE_CANDIDATES if os.path.exists(p)), None)

    NAME_CANDIDATES = [
        "Noto Sans Tamil", "Noto Serif Tamil", "Lohit Tamil",
        "Nirmala UI", "Latha", "Arial Unicode MS", "FreeSerif", "FreeSans",
    ]
    installed = {f.name for f in fm.fontManager.ttflist}
    name_chosen = next((n for n in NAME_CANDIDATES if n in installed), None)

    if font_file:
        fp = FontProperties(fname=font_file)
        plt.rcParams["font.family"] = fp.get_name()
        print(f"Tamil font (file): {font_file}")
    elif name_chosen:
        fp = FontProperties(family=name_chosen)
        plt.rcParams["font.family"] = name_chosen
        print(f"Tamil font (name): {name_chosen}")
    else:
        fp = FontProperties()
        print("Warning: no Tamil font found — labels may render as boxes")

    plt.rcParams["axes.unicode_minus"] = False
    return fp

TAMIL_FP = configure_tamil_font()

# --- v3: one plot style for every figure in the ladder ------------------------
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
Path("outputs").mkdir(exist_ok=True)


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).replace("\ufeff", " ").replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

# --- v3: minimal special-token set -------------------------------------------
# Removed vs v2, deliberately:
#   * dataset identity tags <naladiyar>/<thirukadukam>/<tholkappiyam>
#     -> they let a model win by recognising WHICH text a pair came from
#        (the source-identification confound behind the 0.78 combined
#        pair-classifier) instead of whether verse and urai actually match.
#   * <lb> line-break token
#     -> tokenize_words only emitted it for text that still had newlines. In
#        the grammar probe the ORIGINAL verse kept its newlines while every
#        scrambled variant was rebuilt as a single line, so the original was
#        scored with (n_lines - 1) extra, highly predictable <lb> tokens that
#        no competitor had. That inflated the 19/19 result. Gone.
PAD, UNK, MASK, BOS, EOS = "<pad>", "<unk>", "<mask>", "<bos>", "<eos>"
VERSE_TAG, URAI_TAG = "<verse>", "<urai>"

SPECIAL_TOKENS = [PAD, UNK, MASK, BOS, EOS, VERSE_TAG, URAI_TAG]

DS_COLORS = {
    "naladiyar":        "tab:blue",
    "thirukadukam":     "tab:purple",
    "tholkappiyam_eth": "tab:orange",
    "tholkappiyam_por": "tab:green",
    "tholkappiyam_sol": "tab:red",
}

def dataset_name_from_path(path):
    p = str(path).lower()
    if "thirukadukam" in p: return "thirukadukam"
    if "naladiyar" in p:    return "naladiyar"
    if "eth" in p:          return "tholkappiyam_eth"
    if "por" in p:          return "tholkappiyam_por"
    if "sol" in p:          return "tholkappiyam_sol"
    return Path(path).stem

def tokenize_words(text: str):
    """v3: flat word list. Line structure is NOT encoded (see the <lb> note
    above), so a verse and any re-linearised rewrite of it are tokenized on
    exactly equal terms."""
    text = normalize_text(text).lower()
    tokens = []
    for word in text.split():
        cleaned = re.sub(r"[^\w\u0B80-\u0BFF]+", "", word)
        if cleaned: tokens.append(cleaned)
    return tokens

def encode_tokens(tokens, stoi):
    unk = stoi.get(UNK, 1)
    return [stoi.get(t, unk) for t in tokens]

In [ ]:
DATA_PATHS = [
    "data/naladiyar.jsonl",
    "data/sft_data-eth.jsonl",
    "data/sft_data-por.jsonl",
    "data/sft_data-sol.jsonl",
    "data/thirukadukam_tamilvu.jsonl",
]

def load_rows(paths):
    rows = []
    for path in paths:
        ds  = dataset_name_from_path(path)
        enc = "utf-8-sig" if "naladiyar" in str(path).lower() else "utf-8"
        with open(path, "r", encoding=enc, errors="ignore") as f:
            for line in f:
                if not line.strip(): continue
                obj   = json.loads(line)
                verse = normalize_text(obj.get("verse", ""))
                urai  = normalize_text(obj.get("explanation", obj.get("explaination", obj.get("urai", ""))))
                if verse and urai:
                    rows.append({"dataset": ds, "verse": verse, "urai": urai})
    return rows

all_rows = load_rows(DATA_PATHS)
df_all   = pd.DataFrame(all_rows)
print(f"Total rows: {len(all_rows)}")
print(df_all["dataset"].value_counts())

In [ ]:
# v2 sanity gate: the corrected corpus MUST load 1,262 pairs (393 Naaladiyar).
# If this fails, the data/ folder does not contain the corrected files from
# classical_tamil_verse_urai_corpus.zip - fix that before training anything.
from collections import Counter as _Counter
_c = _Counter(r["dataset"] for r in all_rows)
print("per-dataset:", dict(_c), " total:", len(all_rows))
assert len(all_rows) == 1262, f"expected 1,262 rows, got {len(all_rows)}"
assert _c.get("naladiyar") == 393, f"expected 393 Naaladiyar rows, got {_c.get('naladiyar')}"
print("OK: corrected corpus confirmed (1,262 pairs, 393 Naaladiyar)")

In [ ]:
def build_vocab(rows, min_freq=1):
    counts = Counter()
    for row in rows:
        counts.update(tokenize_words(row["verse"]))
        counts.update(tokenize_words(row["urai"]))
    stoi = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for tok, freq in counts.items():
        if freq >= min_freq and tok not in stoi:
            stoi[tok] = len(stoi)
    return stoi

shared_vocab = build_vocab(all_rows)
shared_itos  = {v: k for k, v in shared_vocab.items()}
print(f"Vocab size: {len(shared_vocab)}")

In [ ]:
def get_nool_rows(nool, rows):
    if nool == "tholkappiyam":
        return [r for r in rows if r["dataset"].startswith("tholkappiyam")]
    if nool == "combined":
        return rows
    return [r for r in rows if r["dataset"] == nool]

NOOLS = ["naladiyar", "thirukadukam", "tholkappiyam"]

In [ ]:
def mask_tokens(ids, mask_id, mask_prob=0.15):
    x = ids.copy(); y = [-100] * len(ids)
    cands = list(range(1, len(ids)))
    if not cands: return x, y
    forced = random.choice(cands)
    for i in cands:
        if random.random() < mask_prob or i == forced:
            y[i] = ids[i]; x[i] = mask_id
    return x, y

class MLMDataset(Dataset):
    def __init__(self, texts, vocab, prefix_toks):
        self.items = []
        for text in texts:
            toks = prefix_toks + tokenize_words(text)
            ids  = encode_tokens(toks, vocab)
            if len(ids) >= 3: self.items.append(ids)
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def mlm_collate(batch, vocab):
    pad_id  = vocab[PAD]; mask_id = vocab[MASK]
    inputs, labels, lengths = [], [], []
    for ids in batch:
        x, y = mask_tokens(ids, mask_id)
        inputs.append(x); labels.append(y); lengths.append(len(x))
    mlen = max(lengths)
    inp_t = torch.full((len(batch), mlen), pad_id, dtype=torch.long)
    lab_t = torch.full((len(batch), mlen), -100,   dtype=torch.long)
    msk_t = torch.zeros(len(batch), mlen, dtype=torch.bool)
    for i, (x, y, l) in enumerate(zip(inputs, labels, lengths)):
        inp_t[i, :l] = torch.tensor(x)
        lab_t[i, :l] = torch.tensor(y)
        msk_t[i, :l] = True
    return inp_t, lab_t, msk_t, torch.tensor(lengths, dtype=torch.long)

## Step 1 — Train 8 separate LSTM encoders

One `LSTMEncoder` per (text, side) — e.g. "Naaladiyar verses" gets its own model, "Naaladiyar commentary" gets a separate one. 256-dim hidden size, 2 layers, bidirectional (reads each sentence forwards *and* backwards), 20 epochs.

**Real result from this run** — masked-word prediction loss, lower is better:

| | Naaladiyar | Thirukadukam | Tholkappiyam | Combined |
|---|---|---|---|---|
| verse | 6.996 | 6.622 | 5.971 | 6.244 |
| urai | 8.257 | 7.674 | 7.593 | 7.828 |

Urai loss is consistently *higher* than verse loss, across every single text — the commentary prose is longer and more varied than the terse verses, so predicting its masked words is the harder task.

In [ ]:
class LSTMEncoder(nn.Module):
    """Unidirectional LSTM. Exposes last hidden state and mean-pooled sentence rep."""
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.15, pad_id=0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=False,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, ids, lengths):
        x = self.embedding(ids)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h_n, _) = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.lm_head(out), out, h_n

    def sentence_reps(self, ids, lengths):
        """(last_hidden, mean_hidden) – both shape [batch, hidden_dim]."""
        with torch.no_grad():
            self.eval()
            _, out, h_n = self.forward(ids, lengths)
            last = h_n[-1]
            mask = torch.arange(out.size(1), device=out.device).unsqueeze(0) < lengths.to(out.device).unsqueeze(1)
            mean = (out * mask.unsqueeze(-1).float()).sum(1) / lengths.float().to(out.device).unsqueeze(1).clamp(min=1)
        return last, mean

In [ ]:
def train_lstm(model, texts, vocab, prefix_toks, name, epochs=20, bs=32, lr=2e-3):
    ds     = MLMDataset(texts, vocab, prefix_toks)
    loader = DataLoader(ds, batch_size=bs, shuffle=True,
                        collate_fn=lambda b: mlm_collate(b, vocab))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    model.to(DEVICE); hist = []
    for ep in range(epochs):
        model.train(); total = n = 0
        for inp, lab, msk, lengths in loader:
            inp = inp.to(DEVICE); lab = lab.to(DEVICE); lengths = lengths.to(DEVICE)
            opt.zero_grad()
            logits, _, _ = model(inp, lengths)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   lab.reshape(-1), ignore_index=-100)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            total += loss.item() * inp.size(0); n += inp.size(0)
        hist.append(total / n)
    print(f"  [{name:45s}]  final_loss={hist[-1]:.4f}")
    model.cpu(); return hist

lstm_verse = {}; lstm_urai = {}; train_hist = {}
print("=== Training LSTM encoders (verse + urai per nool + combined) ===")

for nool in NOOLS + ["combined"]:
    nool_rows = get_nool_rows(nool, all_rows)
    verses = [r["verse"] for r in nool_rows]
    urais  = [r["urai"]  for r in nool_rows]
    print(f"\n[{nool}]  n={len(nool_rows)}")

    mv = LSTMEncoder(len(shared_vocab), pad_id=shared_vocab[PAD])
    mu = LSTMEncoder(len(shared_vocab), pad_id=shared_vocab[PAD])
    hv = train_lstm(mv, verses, shared_vocab, [VERSE_TAG], f"{nool}/verse", epochs=20)
    hu = train_lstm(mu, urais,  shared_vocab, [URAI_TAG],  f"{nool}/urai",  epochs=20)
    lstm_verse[nool] = mv; lstm_urai[nool] = mu
    train_hist[nool] = (hv, hu)

fig, axes = plt.subplots(1, len(NOOLS)+1, figsize=(4*(len(NOOLS)+1), 3), sharey=True)
for ax, nool in zip(axes, NOOLS + ["combined"]):
    hv, hu = train_hist[nool]
    ax.plot(hv, label="verse"); ax.plot(hu, label="urai")
    ax.set_title(nool, fontsize=9); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3); ax.legend(fontsize=7)
axes[0].set_ylabel("Masked-LM loss")
plt.suptitle("LSTM Training Curves", fontsize=11); plt.tight_layout(); plt.show()

## Step 2 — Turn each verse and urai into a single vector

Two ways to collapse a whole sentence into one fixed-size vector:
- **last hidden state** — the model's final "memory" after reading every word
- **mean pooling** — the average of what the model was "thinking" at every single word

Both are carried forward into every test below, so we can compare which pooling method gives a more meaningful representation.

In [ ]:
@torch.no_grad()
def extract_sentence_reps(verse_model, urai_model, rows, vocab, max_n=None):
    verse_model.eval().to(DEVICE); urai_model.eval().to(DEVICE)
    records = []
    for row in (rows[:max_n] if max_n else rows):
        def get(model, text, tag):
            ids = encode_tokens([tag] + tokenize_words(text), vocab)
            ids_t = torch.tensor([ids], dtype=torch.long, device=DEVICE)
            lens_t = torch.tensor([len(ids)], dtype=torch.long, device=DEVICE)
            last, mean = model.sentence_reps(ids_t, lens_t)
            return last[0].cpu().numpy(), mean[0].cpu().numpy()
        v_last, v_mean = get(verse_model, row["verse"], VERSE_TAG)
        u_last, u_mean = get(urai_model,  row["urai"],  URAI_TAG)
        records.append({"dataset": row["dataset"],
                        "verse": row["verse"], "urai": row["urai"],
                        "v_last": v_last, "v_mean": v_mean,
                        "u_last": u_last, "u_mean": u_mean})
    verse_model.cpu(); urai_model.cpu()
    return records

reps_per_nool = {}
for nool in NOOLS + ["combined"]:
    nool_rows = get_nool_rows(nool, all_rows)
    reps_per_nool[nool] = extract_sentence_reps(lstm_verse[nool], lstm_urai[nool], nool_rows, shared_vocab)
    print(f"  [{nool}]  {len(reps_per_nool[nool])} paired reps")


## Test 1 — Do a verse and its own commentary naturally look similar?

**Cosine similarity**: 1.0 = pointing in the exact same direction (very similar), 0 = unrelated, -1 = opposite. **Euclidean distance**: plain straight-line distance, smaller is closer.

**Real result:** cosine similarity between a verse and its *own real* commentary averages only **0.067** (last-hidden) / **0.035** (mean-pool) for Naaladiyar — barely above zero. Thirukadukam is even flatter: **0.074, std=0.000**, the vectors are nearly identical across every Thirukadukam pair, meaning the model isn't meaningfully distinguishing between its verses at all.

**Why this is expected:** the verse-LSTM and urai-LSTM were trained completely separately, with no shared task connecting them — so the possibility of linear gemometrical lineup is limited. This sets up why the next two tests (CCA / KCCA) matter.

Below: the actual top-2 most-similar and bottom-2 least-similar verse–urai pairs by each measure, so you can read real examples instead of just the averages.

In [ ]:
def _trunc(text, n=45):
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n].rstrip() + "…"

from IPython.display import display

def show_top_bottom(recs, scores, metric_name, nool, rep_type, higher_is_more_similar, n=2):
    """Display the n most-similar and n least-similar verse-urai pairs by this measure."""
    df = pd.DataFrame({
        "dataset": [r["dataset"] for r in recs],
        "verse": [_trunc(r["verse"]) for r in recs],
        "urai": [_trunc(r["urai"]) for r in recs],
        metric_name: scores,
    })
    top = df.sort_values(metric_name, ascending=not higher_is_more_similar).head(n).reset_index(drop=True)
    bottom = df.sort_values(metric_name, ascending=higher_is_more_similar).head(n).reset_index(drop=True)
    print(f"  -- [{nool} | {rep_type}] Top {n} MOST similar pairs by {metric_name} --")
    display(top)
    print(f"  -- [{nool} | {rep_type}] Top {n} LEAST similar pairs by {metric_name} --")
    display(bottom)

def cosine_sim(A, B):
    A_n = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    B_n = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return np.sum(A_n * B_n, axis=1)

def euclidean_dist(A, B):
    return np.linalg.norm(A - B, axis=1)

print("=== Cosine Similarity + Euclidean Distance (verse vs urai) ===")
for nool in NOOLS + ["combined"]:
    recs = reps_per_nool[nool]
    for rep_type in ["last", "mean"]:
        V = np.stack([r[f"v_{rep_type}"] for r in recs])
        U = np.stack([r[f"u_{rep_type}"] for r in recs])
        cos  = cosine_sim(V, U)
        eucl = euclidean_dist(V, U)
        print(f"  [{nool:14s} | {rep_type}]  cos: mean={cos.mean():.3f} std={cos.std():.3f}  "
              f"eucl: mean={eucl.mean():.3f} std={eucl.std():.3f}")
        show_top_bottom(recs, cos, "cosine", nool, rep_type, higher_is_more_similar=True)
        show_top_bottom(recs, eucl, "euclidean", nool, rep_type, higher_is_more_similar=False)

# Distribution plots – cosine and euclidean for last/mean, per nool
n_cols = len(NOOLS) + 1
fig, axes = plt.subplots(4, n_cols, figsize=(4*n_cols, 12))
for col, nool in enumerate(NOOLS + ["combined"]):
    recs = reps_per_nool[nool]
    for row_idx, (rep_type, metric, fn, color) in enumerate([
        ("last", "Cosine",    cosine_sim,     "tab:blue"),
        ("last", "Euclidean", euclidean_dist, "tab:orange"),
        ("mean", "Cosine",    cosine_sim,     "tab:green"),
        ("mean", "Euclidean", euclidean_dist, "tab:red"),
    ]):
        V = np.stack([r[f"v_{rep_type}"] for r in recs])
        U = np.stack([r[f"u_{rep_type}"] for r in recs])
        vals = fn(V, U)
        axes[row_idx, col].hist(vals, bins=20, color=color, alpha=0.8, edgecolor="white")
        axes[row_idx, col].set_title(f"{nool}\n{metric} ({rep_type})", fontsize=8)
        axes[row_idx, col].grid(True, alpha=0.3)
plt.suptitle("LSTM Verse–Urai Distance Distributions\n(all nools, last & mean, cosine & euclidean)", fontsize=11)
plt.tight_layout(); plt.show()


## Test 2 — Is there a hidden *straight-line* relationship?

**CCA (Canonical Correlation Analysis)** doesn't ask "are these two vectors already similar", it asks "if I'm allowed to *rotate and rescale* both spaces optimally, can I line them up?" It reports a correlation (0 to 1) for several independent directions ("axes") it finds.

**Real result:** near-perfect scores everywhere: Naaladiyar: **1.000, 1.000, 0.951...**, Thirukadukam: **1.000** across the board, even combined data: **0.992, 0.964, 0.890...**

**Read this carefully:** this does *not* mean the model deeply understands verse–urai meaning. CCA is a very flexible linear technique, and with relatively few examples (as few as 100 for Thirukadukam) compared to how many dimensions the vectors have, it can find a rotation that fits almost any two datasets near-perfectly. It shows a linear mapping *can be found*, not that the model *learned* one during training.

In [ ]:
# ── CCA (Canonical Correlation Analysis) ────────────────────────────
N_COMP = 6
print("=== CCA ===")
for nool in NOOLS + ["combined"]:
    recs = reps_per_nool[nool]
    for rep_type in ["last", "mean"]:
        V = np.stack([r[f"v_{rep_type}"] for r in recs])
        U = np.stack([r[f"u_{rep_type}"] for r in recs])
        n_comp = min(N_COMP, V.shape[1] // 4, U.shape[1] // 4, len(recs) - 1)
        cca = CCA(n_components=n_comp, max_iter=1000)
        Vc, Uc = cca.fit_transform(V, U)
        corrs = [float(np.corrcoef(Vc[:, k], Uc[:, k])[0, 1]) for k in range(n_comp)]
        print(f"  [{nool:14s} | {rep_type}]  CCA r = {[f'{c:.3f}' for c in corrs]}")
        pair_score = Vc[:, 0] * Uc[:, 0]
        show_top_bottom(recs, pair_score, "cca_axis1_alignment", nool, rep_type, higher_is_more_similar=True)

# Bar chart of CCA correlations per nool (mean-pooled)
fig, axes = plt.subplots(1, len(NOOLS)+1, figsize=(4*(len(NOOLS)+1), 4), sharey=True)
for ax, nool in zip(axes, NOOLS + ["combined"]):
    recs = reps_per_nool[nool]
    V = np.stack([r["v_mean"] for r in recs]); U = np.stack([r["u_mean"] for r in recs])
    n_comp = min(N_COMP, V.shape[1]//4, len(recs)-1)
    cca = CCA(n_components=n_comp, max_iter=1000)
    Vc, Uc = cca.fit_transform(V, U)
    corrs = [float(np.corrcoef(Vc[:, k], Uc[:, k])[0, 1]) for k in range(n_comp)]
    ax.bar(range(1, n_comp+1), corrs, color="tab:purple", alpha=0.8)
    ax.set_title(nool, fontsize=9); ax.set_xlabel("CCA axis"); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("Canonical correlation")
plt.suptitle("CCA canonical correlations – mean-pooled LSTM representations", fontsize=11)
plt.tight_layout(); plt.show()


## Test 3 — Is there a hidden *curved* (non-linear) relationship instead?

**KCCA (Kernel CCA)** is the same idea as CCA, but first passes the data through an RBF kernel — a mathematical trick that can capture bent/curved relationships a straight line can't.

**Real result:** correlations collapse to near zero or even negative — Naaladiyar: **0.019, 0.026, 0.116...**, Tholkappiyam: **-0.283, -0.121...**, combined swings wildly between **-0.908** and **0.685** (unstable — not a real signal).

**Why this is actually informative:** the previous cell's CCA scores (near 1.0) were suspiciously high and KCCA finding *nothing* here confirms that suspicion. If verse-meaning and urai-meaning were genuinely related, a *more powerful, more flexible* method (KCCA) should find that relationship at least as well as the simpler one (CCA). Instead it finds *less* — strong evidence the CCA scores were fitting noise in a small sample, not a real signal.

In [ ]:
# ── KCCA (Kernel CCA with RBF kernel) ─────────────────────────────────
def kernel_cca(X, Y, n_components=6, reg=1e-3, return_proj=False):
    n = X.shape[0]
    gx = 1.0 / X.shape[1]; gy = 1.0 / Y.shape[1]
    Kx = rbf_kernel(X, gamma=gx); Ky = rbf_kernel(Y, gamma=gy)
    H  = np.eye(n) - np.ones((n, n)) / n
    Kxc = H @ Kx @ H; Kyc = H @ Ky @ H
    A   = Kxc @ Kyc
    B   = (Kxc + reg * np.eye(n)) @ (Kyc + reg * np.eye(n))
    vals, vecs = np.linalg.eig(np.linalg.solve(B + 1e-9*np.eye(n), A))
    vals = np.real(vals); vecs = np.real(vecs)
    idx  = np.argsort(vals)[::-1][:n_components]
    alphas = vecs[:, idx]
    Vk = Kxc @ alphas
    reg_B_inv = np.linalg.solve((Kyc + reg*np.eye(n)) @ (Kxc + reg*np.eye(n)) + 1e-9*np.eye(n),
                                 Kyc @ alphas)
    Uk = Kyc @ reg_B_inv
    corrs = []
    for k in range(n_components):
        c = np.corrcoef(Vk[:, k], Uk[:, k])[0, 1]
        corrs.append(float(np.real(c)) if np.isfinite(c) else 0.0)
    if return_proj:
        return corrs, Vk, Uk
    return corrs

print("=== KCCA (RBF kernel) ===")
for nool in NOOLS + ["combined"]:
    recs = reps_per_nool[nool]
    for rep_type in ["last", "mean"]:
        V = np.stack([r[f"v_{rep_type}"] for r in recs])
        U = np.stack([r[f"u_{rep_type}"] for r in recs])
        n_comp = min(N_COMP, len(recs) - 1)
        corrs, Vk, Uk = kernel_cca(V, U, n_components=n_comp, return_proj=True)
        print(f"  [{nool:14s} | {rep_type}]  KCCA r = {[f'{c:.3f}' for c in corrs]}")
        pair_score = np.real(Vk[:, 0] * Uk[:, 0])
        show_top_bottom(recs, pair_score, "kcca_axis1_alignment", nool, rep_type, higher_is_more_similar=True)

# Compare CCA vs KCCA for mean-pooled, all nools
fig, axes = plt.subplots(1, len(NOOLS)+1, figsize=(4*(len(NOOLS)+1), 4), sharey=True)
for ax, nool in zip(axes, NOOLS + ["combined"]):
    recs = reps_per_nool[nool]
    V = np.stack([r["v_mean"] for r in recs]); U = np.stack([r["u_mean"] for r in recs])
    n_comp = min(N_COMP, V.shape[1]//4, len(recs)-1)
    cca = CCA(n_components=n_comp, max_iter=1000)
    Vc, Uc = cca.fit_transform(V, U)
    cca_r  = [float(np.corrcoef(Vc[:, k], Uc[:, k])[0, 1]) for k in range(n_comp)]
    kcca_r = kernel_cca(V, U, n_components=n_comp)
    x = np.arange(1, n_comp+1)
    ax.bar(x-0.2, cca_r,  0.38, label="CCA",  color="tab:purple", alpha=0.8)
    ax.bar(x+0.2, kcca_r, 0.38, label="KCCA", color="tab:red",    alpha=0.8)
    ax.set_title(nool, fontsize=9); ax.set_xlabel("Axis"); ax.set_ylim(0, 1)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("Canonical correlation")
plt.suptitle("CCA vs KCCA – mean-pooled LSTM representations", fontsize=11)
plt.tight_layout(); plt.show()


## Test 4 — Can a small classifier learn to spot a real match?

Build one feature vector from *both* sides at once — `[v, u, |v−u|, v⊙u]` (the raw vectors, their difference, and their element-wise product) — and train a small 2-layer network to say "real pair" vs "random pair" (0.500 = coin-flip / chance level).

**Real result:**

| | Naaladiyar | Thirukadukam | Tholkappiyam | Combined |
|---|---|---|---|---|
| last hidden | 0.500 | 0.500 | 0.616 | 0.779 |
| mean pool | 0.500 | 0.500 | 0.653 | 0.795 |

Within a single text, the classifier is stuck at exactly chance (**0.500**): it genuinely cannot tell a real verse–urai pair from a randomly swapped one. Once all texts are combined, accuracy jumps to **~0.78–0.80**. That jump shows that at last it collapses to corpus source rather than actual semantic meaning of the verses and it's corresponding urai

In [ ]:
# ── Siamese-style pair classifier  [v, u, |v−u|, v⊙u] → match / no-match ──
class PairClassifier(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64),      nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, v, u):
        return self.net(torch.cat([v, u, (v-u).abs(), v*u], dim=-1)).squeeze(-1)

class PairDataset(Dataset):
    def __init__(self, V, U, neg_ratio=1):
        self.items = []
        n = len(V)
        for i in range(n):
            self.items.append((V[i], U[i], 1.0))
            for _ in range(neg_ratio):
                j = i
                while j == i: j = random.randrange(n)
                self.items.append((V[i], U[j], 0.0))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        v, u, y = self.items[i]
        return torch.tensor(v, dtype=torch.float), torch.tensor(u, dtype=torch.float), torch.tensor(y)

def run_pair_cls(V, U, rep_type, nool, epochs=20, bs=32, lr=1e-3):
    dim = V.shape[1]
    clf = PairClassifier(dim).to(DEVICE)
    ds  = PairDataset(V, U, neg_ratio=1)
    loader = DataLoader(ds, batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-2)
    for _ in range(epochs):
        clf.train()
        for vb, ub, yb in loader:
            vb = vb.to(DEVICE); ub = ub.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad()
            F.binary_cross_entropy_with_logits(clf(vb, ub), yb).backward()
            opt.step()
    clf.eval()
    preds, labels = [], []
    with torch.no_grad():
        for vb, ub, yb in loader:
            preds.extend((torch.sigmoid(clf(vb.to(DEVICE), ub.to(DEVICE))) > 0.5).cpu().tolist())
            labels.extend(yb.tolist())
    acc = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    print(f"  [{nool:14s} | {rep_type}]  pair-cls accuracy = {acc:.3f}")

    with torch.no_grad():
        vt = torch.tensor(V, dtype=torch.float, device=DEVICE)
        ut = torch.tensor(U, dtype=torch.float, device=DEVICE)
        true_pair_conf = torch.sigmoid(clf(vt, ut)).cpu().numpy()
    return acc, true_pair_conf

print("=== Siamese-style Pair Classifier [v, u, |v−u|, v⊙u] ===")
accs = {}
for nool in NOOLS + ["combined"]:
    recs = reps_per_nool[nool]
    accs[nool] = {}
    for rep_type in ["last", "mean"]:
        V = np.stack([r[f"v_{rep_type}"] for r in recs])
        U = np.stack([r[f"u_{rep_type}"] for r in recs])
        acc, true_pair_conf = run_pair_cls(V, U, rep_type, nool)
        accs[nool][rep_type] = acc
        show_top_bottom(recs, true_pair_conf, "match_confidence", nool, rep_type, higher_is_more_similar=True)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(NOOLS) + 1)
labels = NOOLS + ["combined"]
ax.bar(x - 0.2, [accs[n]["last"] for n in labels], 0.38, label="last hidden", color="tab:blue", alpha=0.8)
ax.bar(x + 0.2, [accs[n]["mean"] for n in labels], 0.38, label="mean pooled", color="tab:orange", alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="random baseline")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Accuracy")
ax.set_title("Pair Classifier Accuracy [v, u, |v−u|, v⊙u]\nall nools")
ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


## Test 5 — See it on a map

A 2D projection (t-SNE) of every verse/urai vector, coloured by which text it came from. Verses are circles (●), commentary is triangles (▲).

**What to look for on screen:** points cluster tightly by *colour* (source text) rather than by verse-matching-its-own-urai. That visual clustering tells the same story as every test above, in picture form: the model organised its number-space around "which text is this from," not "which commentary explains this specific verse." All the verses and urai were clustered seperately for individual text and for combination of texts, the clustering was visbile based on the origin of the verse-urai from a text.

In [ ]:
print("=== t-SNE of LSTM sentence representations ===")
for nool in NOOLS + ["combined"]:
    recs = reps_per_nool[nool]
    for rep_type in ["last", "mean"]:
        entries = (
            [{"ds": r["dataset"], "side": "verse", "vec": r[f"v_{rep_type}"]} for r in recs] +
            [{"ds": r["dataset"], "side": "urai",  "vec": r[f"u_{rep_type}"]} for r in recs]
        )
        X   = np.stack([e["vec"] for e in entries])
        perp = min(30, max(5, len(entries) // 10))
        X2d  = TSNE(n_components=2, perplexity=perp, random_state=SEED,
                    init="pca", learning_rate="auto").fit_transform(X)
        edf  = pd.DataFrame(entries); edf["x"] = X2d[:, 0]; edf["y"] = X2d[:, 1]
        type_m = {"verse": "o", "urai": "^"}
        fig, ax = plt.subplots(figsize=(10, 7))
        for ds in sorted(edf["ds"].unique()):
            for side in ["verse", "urai"]:
                sub = edf[(edf["ds"] == ds) & (edf["side"] == side)]
                ax.scatter(sub["x"], sub["y"], c=DS_COLORS.get(ds, "gray"),
                           marker=type_m[side], s=35, alpha=0.75, label=f"{ds}|{side}")
        ax.set_title(f"LSTM t-SNE – {nool} | {rep_type}\nVerse (●) vs Urai (▲)")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
        ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()